# SEM Image Upscaling Notebook  📈🖼️  
This Jupyter notebook lets you upscale grayscale Scanning Electron Microscope (SEM) TIFF images with several off‑the‑shelf methods (SwinIR, Real‑ESRGAN, BM3D + Lanczos, etc.).  
Edit the paths and parameters in **Step 1** and run each cell in order.

In [ ]:
# --- 🔧 Step 1: Configure paths & parameters ---
from pathlib import Path
INPUT_DIR = Path('examples')  # bundled synthetic demo image
OUTPUT_DIR = Path('output_sem_sr').expanduser()
METHOD     = 'swinir-real'   # 'swinir-real' | 'swinir-classical' | 'realesrgan' | 'lanczos' | 'bm3d-lanczos'
SCALE      = 4               # 2 or 4
BM3D_SIGMA = 25              # noise σ (uint8 range) used only for bm3d-lanczos

print('Input folder :', INPUT_DIR.resolve())
print('Output folder:', OUTPUT_DIR.resolve())


In [ ]:
# --- 📦 Step 2: Install required libraries (run once) ---
!pip install -q --upgrade opencv-python numpy pillow tifffile torch torchvision bm3d         realesrgan basicsr


In [ ]:
# --- 📚 Step 3: Imports & helper functions ---
import cv2, torch, tifffile, numpy as np, os, sys
from PIL import Image
from pathlib import Path

def read_sem_tiff(path, meta_strip_ratio: float = 0.1):
    img = tifffile.imread(str(path))
    if img.ndim == 3 and img.shape[2] == 1:
        img = img[:, :, 0]
    meta_h = int(img.shape[0] * meta_strip_ratio)
    meta = img[-meta_h:] if meta_h else np.zeros((0, *img.shape[1:]), img.dtype)
    content = img[:-meta_h] if meta_h else img
    return content, meta

def save_uint8(arr, path):
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    if arr.ndim == 2:
        Image.fromarray(arr, mode='L').save(path)
    else:
        Image.fromarray(arr).save(path)

def to_float01(arr):
    arr = arr.astype(np.float32)
    if arr.max() > 1: arr /= 255.0
    return arr

def bm3d_denoise(gray, sigma=25/255.0):
    from bm3d import bm3d_gray
    return np.clip(bm3d_gray(gray, sigma_psd=sigma), 0.0, 1.0)

def lanczos4(img, scale):
    h, w = img.shape[:2]
    return cv2.resize(img, (w*scale, h*scale), interpolation=cv2.INTER_LANCZOS4)

def upscale_swinir(img, scale, variant='real'):
    assert scale in (2,4)
    name = f'swinir_{variant}_sr_x{scale}'
    model = torch.hub.load('JingyunLiang/SwinIR', name, pretrained=True).eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    if img.ndim == 2:
        inp = np.stack([img]*3, axis=2)
    else:
        inp = img[:,:,:3]
    inp = torch.from_numpy(inp).permute(2,0,1).float().unsqueeze(0)/255.0
    with torch.no_grad():
        out = model(inp.to(device)).cpu().clamp(0,1)
    out = (out.squeeze(0).permute(1,2,0).numpy()*255.0)
    if img.ndim == 2:
        out = cv2.cvtColor(out.astype(np.uint8), cv2.COLOR_BGR2GRAY)
    return out

def upscale_realesrgan(img, scale):
    from realesrgan import RealESRGANer
    from realesrgan.archs.rrdbnet_arch import RRDBNet
    model = RRDBNet(3,3,64,23,32,scale=scale)
    model_path = RealESRGANer.get_default_model_path(scale=scale)
    upsampler = RealESRGANer(scale=scale, model_path=model_path, model=model,
                             tile=0, half=False,
                             device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if img.ndim==2 else img[:,:,:3]
    sr,_ = upsampler.enhance(bgr)
    return cv2.cvtColor(sr, cv2.COLOR_BGR2GRAY) if img.ndim==2 else sr


In [ ]:
# --- 🚀 Step 4: Batch process all TIFFs ---
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
files = sorted(list(INPUT_DIR.glob('*.tif')) + list(INPUT_DIR.glob('*.tiff')))
print(f'Found {len(files)} file(s).')

for fp in files:
    print('•', fp.name)
    content, meta = read_sem_tiff(fp)
    gray = content if content.ndim==2 else cv2.cvtColor(content, cv2.COLOR_BGR2GRAY)

    if METHOD.startswith('bm3d'):
        gray = (bm3d_denoise(to_float01(gray), sigma=BM3D_SIGMA/255.0)*255).astype(np.uint8)

    if METHOD == 'swinir-real':
        up = upscale_swinir(gray, SCALE, variant='real')
    elif METHOD == 'swinir-classical':
        up = upscale_swinir(gray, SCALE, variant='classical')
    elif METHOD == 'realesrgan':
        up = upscale_realesrgan(gray, SCALE)
    elif METHOD in ('lanczos','bm3d-lanczos'):
        up = lanczos4(gray, SCALE)
    else:
        raise ValueError('Unknown METHOD')

    # stitch metadata bar back
    if meta.size:
        meta_up = cv2.resize(meta, (up.shape[1], meta.shape[0]*SCALE), interpolation=cv2.INTER_NEAREST)
        up_full = np.vstack([up, meta_up])
    else:
        up_full = up

    out_path = OUTPUT_DIR / f'{fp.stem}_{METHOD}_x{SCALE}.png'
    save_uint8(up_full, out_path)
    print('   ↳ saved', out_path)


In [ ]:
# --- 👀 Step 5: Display the last result (if Jupyter supports inline) ---
from IPython.display import display
from PIL import Image
if files:
    display(Image.open(out_path))
